In [ ]:
# Import necessary libraries
import os
import json
import time
import random
import logging
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any, Optional
from IPython.display import display, HTML

# Selenium imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("azure_scraper_notebook")

In [ ]:
class PartnerData:
    """Class to store partner data."""
    
    def __init__(
        self, 
        name: str, 
        url: str, 
        description: str = "", 
        logo_url: str = "", 
        location: str = "",
        capabilities: List[str] = None
    ):
        self.name = name
        self.url = url
        self.description = description
        self.logo_url = logo_url
        self.location = location
        self.capabilities = capabilities or []
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary."""
        return {
            "name": self.name,
            "url": self.url,
            "description": self.description,
            "logo_url": self.logo_url,
            "location": self.location,
            "capabilities": self.capabilities
        }
    
    def __repr__(self) -> str:
        return f"PartnerData(name='{self.name}', location='{self.location}', capabilities={self.capabilities})"

In [ ]:
class AzurePartnerScraperSelenium:
    """
    Scraper for Azure partners directory using Selenium.
    """
    
    def __init__(self, headless: bool = False):
        """
        Initialize the scraper.
        
        Args:
            headless: Whether to run the browser in headless mode
        """
        logger.info("Initializing Azure Partner Scraper with Selenium")
        
        # Set up Chrome options
        chrome_options = webdriver.ChromeOptions()
        if headless:
            chrome_options.add_argument("--headless")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        
        # Initialize the driver
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=chrome_options
        )
        
        # Set default URL
        self.base_url = "https://appsource.microsoft.com/en-us/partners"
        
        logger.info("Selenium driver initialized")
    
    def load_partners_directory(self) -> None:
        """Load the partners directory page."""
        self.load_url(self.base_url)
    
    def load_url(self, url: str) -> None:
        """
        Load a specific URL.
        
        Args:
            url: URL to load
        """
        logger.info(f"Loading URL: {url}")
        self.driver.get(url)
        
        # Wait for the page to load
        try:
            WebDriverWait(self.driver, 30).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            logger.info("Page loaded successfully")
        except TimeoutException:
            logger.warning("Timeout waiting for page to load")
    
    def _similar_text(self, text1: str, text2: str, threshold: float = 0.7) -> bool:
        """
        Check if two texts are similar using a simple similarity measure.
        
        Args:
            text1: First text
            text2: Second text
            threshold: Similarity threshold (0-1)
            
        Returns:
            True if texts are similar, False otherwise
        """
        # Convert to lowercase
        text1 = text1.lower()
        text2 = text2.lower()
        
        # Check if one is a substring of the other
        if text1 in text2 or text2 in text1:
            return True
        
        # Check if they share enough common words
        words1 = set(text1.split())
        words2 = set(text2.split())
        
        if not words1 or not words2:
            return False
        
        common_words = words1.intersection(words2)
        similarity = len(common_words) / max(len(words1), len(words2))
        
        return similarity >= threshold

In [ ]:
def apply_filter(self, filter_type: str, filter_value: str) -> None:
    """
    Apply a filter to the partners directory page.
    
    Args:
        filter_type: Type of filter (e.g., "location", "capability", "industry")
        filter_value: Value to filter by
    """
    logger.info(f"Applying filter: {filter_type} = {filter_value}")
    
    try:
        # Wait for the page to load completely
        time.sleep(3)
        
        # Special handling for location filter
        if filter_type.lower() == "location":
            try:
                # Look for the location filter container
                location_container = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".filters-location-container"))
                )
                
                # Find and click the "Select location" link
                location_edit_link = location_container.find_element(By.CSS_SELECTOR, ".filter-location-edit-link")
                location_edit_link.click()
                logger.info("Clicked 'Select location' link")
                time.sleep(2)
                
                # Wait for the location dialog to appear
                location_dialog = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".ms-Dialog, .ms-Panel, [role='dialog']"))
                )
                
                # Try to find the search box in the dialog
                search_boxes = location_dialog.find_elements(By.CSS_SELECTOR, 
                    "input[type='text'], .ms-SearchBox-field, [role='searchbox']")
                
                if search_boxes:
                    search_box = search_boxes[0]
                    search_box.clear()
                    search_box.send_keys(filter_value)
                    search_box.send_keys(Keys.ENTER)
                    logger.info(f"Entered location search: {filter_value}")
                    time.sleep(2)
                
                # Look for matching location options
                location_options = location_dialog.find_elements(By.XPATH, 
                    f"//*[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{filter_value.lower()}')]")
                
                if location_options:
                    for option in location_options:
                        if option.is_displayed() and option.is_enabled():
                            option.click()
                            logger.info(f"Selected location: {option.text}")
                            time.sleep(2)
                            
                            # Look for Apply/OK button
                            apply_buttons = location_dialog.find_elements(By.CSS_SELECTOR, 
                                "button.ms-Button--primary, [aria-label='Apply'], [aria-label='OK']")
                            
                            if apply_buttons:
                                apply_buttons[0].click()
                                logger.info("Clicked Apply button for location filter")
                                time.sleep(3)
                                return
                
                # If we couldn't find or select a location, try to close the dialog
                close_buttons = location_dialog.find_elements(By.CSS_SELECTOR, 
                    "button.ms-Dialog-button--close, .ms-Panel-closeButton, [aria-label='Close']")
                
                if close_buttons:
                    close_buttons[0].click()
                    logger.info("Closed location dialog without selecting")
                    time.sleep(1)
            
            except Exception as e:
                logger.warning(f"Error applying location filter: {e}")
                # Continue with the general filter approach
        
        # For other filter types, use the navigation filter structure
        # Look for filter in the navigation menu
        nav_elements = self.driver.find_elements(By.CSS_SELECTOR, "nav[role='navigation']")
        
        if nav_elements:
            nav = nav_elements[0]
            
            # Find all filter groups
            groups = nav.find_elements(By.CSS_SELECTOR, ".ms-Nav-group")
            
            for group in groups:
                # Find filter items within the group
                items = group.find_elements(By.CSS_SELECTOR, ".ms-Nav-navItem")
                
                for item in items:
                    # Get the filter name
                    name_element = item.find_element(By.CSS_SELECTOR, ".ms-TooltipHost span") if item.find_elements(By.CSS_SELECTOR, ".ms-TooltipHost span") else None
                    
                    if name_element:
                        filter_name = name_element.text.strip()
                        
                        # Check if this is the filter we're looking for
                        if (filter_type.lower() in filter_name.lower() or 
                            self._similar_text(filter_type, filter_name)):
                            
                            # Get the chevron button to expand the filter
                            chevron_button = item.find_element(By.CSS_SELECTOR, ".ms-Nav-chevronButton") if item.find_elements(By.CSS_SELECTOR, ".ms-Nav-chevronButton") else None
                            
                            if chevron_button:
                                # Check if already expanded
                                expanded = chevron_button.get_attribute('aria-expanded')
                                
                                if expanded != 'true':
                                    # Click to expand
                                    chevron_button.click()
                                    logger.info(f"Expanded filter: {filter_name}")
                                    time.sleep(2)
                                
                                # Now look for the filter value in the expanded section
                                # First, find the parent composite link
                                composite_link = item.find_element(By.CSS_SELECTOR, ".ms-Nav-compositeLink")
                                
                                # Find the expanded content - it should be a sibling or child of the composite link
                                expanded_content = None
                                
                                # Try to find expanded content as a sibling
                                parent = composite_link.find_element(By.XPATH, "./..")
                                expanded_elements = parent.find_elements(By.CSS_SELECTOR, 
                                    ".ms-Nav-navItems, .ms-Nav-group, [role='group']")
                                
                                if expanded_elements:
                                    expanded_content = expanded_elements[0]
                                
                                if expanded_content:
                                    # Look for checkboxes or links with the filter value
                                    filter_options = expanded_content.find_elements(By.XPATH, 
                                        f".//*[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{filter_value.lower()}')]")
                                    
                                    if filter_options:
                                        for option in filter_options:
                                            if option.is_displayed():
                                                # Scroll to the option
                                                self.driver.execute_script("arguments[0].scrollIntoView();", option)
                                                time.sleep(1)
                                                
                                                # Click the option
                                                option.click()
                                                logger.info(f"Selected filter value: {option.text}")
                                                time.sleep(3)
                                                return
                                
                                # If we couldn't find or select a value, collapse the filter
                                if expanded == 'false':
                                    chevron_button.click()
                                    logger.info(f"Collapsed filter: {filter_name}")
                                    time.sleep(1)
        
        # If we couldn't find the filter using the navigation structure, fall back to the previous approach
        # First, try to find filter buttons by their text content
        filter_buttons = self.driver.find_elements(By.XPATH, 
            f"//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{filter_type.lower()}')]")
        
        # If no buttons found by text, try common filter button selectors
        if not filter_buttons:
            filter_selectors = [
                "button.filter-button", 
                ".filter-dropdown",
                ".ms-Dropdown",
                ".ms-ComboBox",
                ".ms-Dropdown-container",
                ".ms-Dropdown-title",
                ".ms-Button",
                "[role='combobox']",
                "[aria-haspopup='listbox']",
                f"[data-automation-id*='{filter_type}']",
                f"[id*='{filter_type}']",
                f"[aria-label*='{filter_type}']"
            ]
            
            for selector in filter_selectors:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for element in elements:
                    # Check if the element text contains the filter type
                    element_text = element.text.lower()
                    if filter_type.lower() in element_text or self._similar_text(filter_type, element_text):
                        filter_buttons.append(element)
                
                if filter_buttons:
                    break
        
        # If still no filter buttons found, try to find any clickable elements that might be filters
        if not filter_buttons:
            logger.info(f"Searching for any potential filter elements for {filter_type}")
            
            # Look for elements with filter-related classes or attributes
            potential_filters = self.driver.find_elements(By.CSS_SELECTOR, 
                "[class*='filter'], [class*='dropdown'], [class*='menu'], [class*='select'], button, [role='button']")
            
            for element in potential_filters:
                try:
                    element_text = element.text.lower()
                    element_html = element.get_attribute('outerHTML').lower()
                    
                    # Check if element might be related to our filter type
                    if (filter_type.lower() in element_text or 
                        filter_type.lower() in element_html or
                        self._similar_text(filter_type, element_text)):
                        
                        filter_buttons.append(element)
                except:
                    continue
        
        if not filter_buttons:
            logger.warning(f"Could not find filter '{filter_type}' with value '{filter_value}'")
            return
        
        # Try to click each potential filter button until we find one that works
        filter_applied = False
        for button in filter_buttons:
            try:
                # Scroll to the button to make it visible
                self.driver.execute_script("arguments[0].scrollIntoView();", button)
                time.sleep(1)
                
                # Click the filter button
                button.click()
                logger.info(f"Clicked filter button for {filter_type}")
                time.sleep(2)
                
                # Now look for the filter value in the dropdown or list
                filter_options = []
                
                # Try different selectors for filter options
                option_selectors = [
                    ".ms-Dropdown-item", 
                    ".ms-ComboBox-option",
                    ".dropdown-item",
                    ".filter-option",
                    "li",
                    "[role='option']",
                    "[role='menuitem']",
                    "[class*='option']",
                    "[class*='item']"
                ]
                
                for selector in option_selectors:
                    options = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    if options:
                        filter_options = options
                        break
                
                # If no options found with CSS selectors, try XPath to find any elements with the filter value
                if not filter_options:
                    filter_options = self.driver.find_elements(By.XPATH, 
                        f"//*[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{filter_value.lower()}')]")
                
                # Try to click the option with the matching filter value
                for option in filter_options:
                    option_text = option.text.strip().lower()
                    if (filter_value.lower() in option_text or 
                        self._similar_text(filter_value, option_text)):
                        
                        # Scroll to the option to make it visible
                        self.driver.execute_script("arguments[0].scrollIntoView();", option)
                        time.sleep(1)
                        
                        # Click the option
                        option.click()
                        logger.info(f"Selected filter value: {filter_value}")
                        filter_applied = True
                        break
                
                if filter_applied:
                    # Wait for the page to update after applying the filter
                    time.sleep(3)
                    break
                else:
                    # If we couldn't find the value, click the button again to close the dropdown
                    button.click()
                    time.sleep(1)
            
            except Exception as e:
                logger.warning(f"Error applying filter {filter_type}={filter_value}: {e}")
                # Try the next button
                continue
        
        if not filter_applied:
            logger.warning(f"Could not apply filter {filter_type}={filter_value}")
    
    except Exception as e:
        logger.error(f"Error applying filter {filter_type}={filter_value}: {e}")

# Add the method to the class
AzurePartnerScraperSelenium.apply_filter = apply_filter

In [ ]:
def extract_partners_from_page(self) -> List[PartnerData]:
    """
    Extract partner data from the current page.
    
    Returns:
        List of PartnerData objects
    """
    partners = []
    
    try:
        # Wait for the page to load completely
        time.sleep(3)
        
        # First, check if we have partner cards with role="listitem"
        partner_cards = self.driver.find_elements(By.CSS_SELECTOR, "[role='listitem']")
        
        # If no cards found with role="listitem", try other selectors
        if not partner_cards:
            # Try different selectors for partner cards based on HTML analysis
            selectors = [
                ".card", 
                ".tile-container",
                ".partner-card", 
                ".listing-item", 
                ".result-item",
                ".ms-List-cell", 
                ".ms-DocumentCard", 
                ".ms-StackItem"
            ]
            
            for selector in selectors:
                cards = self.driver.find_elements(By.CSS_SELECTOR, selector)
                if cards:
                    partner_cards = cards
                    logger.info(f"Found {len(cards)} partner cards with selector: {selector}")
                    break
        
        # If still no cards found, try to find any div elements that might be partner cards
        if not partner_cards:
            logger.info("Searching for any potential partner card elements")
            
            # Look for divs with multiple text elements and links
            divs = self.driver.find_elements(By.TAG_NAME, "div")
            
            for div in divs:
                try:
                    # Check if div has links and text content
                    links = div.find_elements(By.TAG_NAME, "a")
                    paragraphs = div.find_elements(By.TAG_NAME, "p")
                    
                    if links and paragraphs and len(paragraphs) >= 2:
                        # This might be a partner card
                        partner_cards.append(div)
                except:
                    continue
            
            if partner_cards:
                logger.info(f"Found {len(partner_cards)} potential partner cards by analyzing div elements")
        
        if not partner_cards:
            logger.warning("Could not find any partner cards on the page")
            return []
        
        # Set a timeout for processing all cards to prevent hanging
        start_time = time.time()
        max_processing_time = 60  # Maximum time in seconds to spend processing cards
        
        logger.info(f"Processing {len(partner_cards)} partner cards")
        
        # Process each partner card
        for i, card in enumerate(partner_cards):
            # Check if we've exceeded the maximum processing time
            if time.time() - start_time > max_processing_time:
                logger.warning(f"Reached maximum processing time after processing {i} cards")
                break
            
            try:
                # Extract partner data from the card
                partner = PartnerData(name="", url="")
                
                # Extract partner name
                name_elements = card.find_elements(By.CSS_SELECTOR, 
                    "h1, h2, h3, h4, h5, h6, .partner-name, .title, [class*='title'], [class*='name'], strong, b")
                
                if name_elements:
                    partner.name = name_elements[0].text.strip()
                else:
                    # Try to find any text that might be a name
                    paragraphs = card.find_elements(By.TAG_NAME, "p")
                    if paragraphs:
                        partner.name = paragraphs[0].text.strip()
                
                # Skip if no name found
                if not partner.name:
                    continue
                
                # Extract partner URL
                link_elements = card.find_elements(By.TAG_NAME, "a")
                
                for link in link_elements:
                    href = link.get_attribute("href")
                    if href and "appsource.microsoft.com" in href and "/product/" in href:
                        partner.url = href
                        break
                
                # If no product URL found, use any URL
                if not partner.url and link_elements:
                    partner.url = link_elements[0].get_attribute("href") or ""
                
                # Extract partner location
                location_elements = card.find_elements(By.XPATH, 
                    ".//*[contains(text(), ',')]")  # Locations often have commas (City, Country)
                
                if location_elements:
                    for element in location_elements:
                        text = element.text.strip()
                        # Check if this looks like a location (contains a comma and not too long)
                        if "," in text and len(text) < 100:
                            partner.location = text
                            break
                
                # Extract partner description
                description_elements = card.find_elements(By.CSS_SELECTOR, 
                    ".description, [class*='description'], [class*='overview'], p")
                
                if description_elements:
                    # Find the longest paragraph that's likely to be a description
                    descriptions = []
                    for element in description_elements:
                        text = element.text.strip()
                        if text and len(text) > 20 and "..." in text:  # Descriptions often end with ellipsis
                            descriptions.append(text)
                        elif text and len(text) > 50:  # Or they're just longer text
                            descriptions.append(text)
                    
                    if descriptions:
                        # Use the longest description
                        partner.description = max(descriptions, key=len)
                
                # Extract partner capabilities
                # Look for lists of technologies or capabilities
                capability_elements = card.find_elements(By.CSS_SELECTOR, 
                    ".tag, [class*='tag'], [class*='pill'], [class*='chip'], [class*='badge']")
                
                if capability_elements:
                    for element in capability_elements:
                        text = element.text.strip()
                        if text and text not in ["Contact me", "Next", "Previous"]:
                            partner.capabilities.append(text)
                
                # If no capabilities found with specific selectors, look for any short text elements
                if not partner.capabilities:
                    # Get all text elements
                    all_text_elements = []
                    for element in card.find_elements(By.XPATH, ".//*"):
                        try:
                            text = element.text.strip()
                            if text and 3 <= len(text) <= 30 and text not in ["Contact me", "Next", "Previous"]:
                                all_text_elements.append(text)
                        except:
                            continue
                    
                    # Filter out elements that are likely to be capabilities
                    known_capabilities = [
                        "Azure", "Office 365", "Dynamics 365", "Microsoft 365", "Power BI", 
                        "SharePoint", "Teams", "Exchange", "SQL", "Windows", "Developer Tools"
                    ]
                    
                    for text in all_text_elements:
                        if text in known_capabilities or any(cap in text for cap in known_capabilities):
                            partner.capabilities.append(text)
                
                # Add the partner to the list if we have at least a name
                if partner.name:
                    partners.append(partner)
                    logger.info(f"Extracted partner: {partner.name}")
            
            except Exception as e:
                logger.warning(f"Error extracting partner data from card {i}: {e}")
                continue
        
        logger.info(f"Successfully extracted {len(partners)} partners from the page")
        
    except Exception as e:
        logger.error(f"Error extracting partners from page: {e}")
    
    return partners

# Add the method to the class
AzurePartnerScraperSelenium.extract_partners_from_page = extract_partners_from_page

In [ ]:
def go_to_next_page(self) -> bool:
    """
    Navigate to the next page of partners.
    
    Returns:
        True if successfully navigated to the next page, False otherwise
    """
    logger.info("Attempting to navigate to the next page")
    
    try:
        # Wait for the page to load completely
        time.sleep(3)
        
        # Look for next button with icon character (from HTML analysis)
        next_buttons = self.driver.find_elements(By.XPATH, "//button[contains(text(), '\ue76c')]")
        
        if not next_buttons:
            # Try to find next button by aria-label
            next_buttons = self.driver.find_elements(By.CSS_SELECTOR, 
                "[aria-label*='Next'], [aria-label*='next'], [title*='Next'], [title*='next']")
        
        if not next_buttons:
            # Try to find by common next button classes
            next_selectors = [
                ".ms-Button--icon[title*='next']",
                ".ms-Button--icon[aria-label*='next']",
                ".ms-Pagination-nextPage",
                "[data-automation-key='nextPage']",
                ".next-page",
                ".pagination-next",
                "[aria-label='Next page']"
            ]
            
            for selector in next_selectors:
                buttons = self.driver.find_elements(By.CSS_SELECTOR, selector)
                if buttons:
                    next_buttons = buttons
                    break
        
        if not next_buttons:
            # As a last resort, look for any button with next-like text
            all_buttons = self.driver.find_elements(By.TAG_NAME, "button")
            for button in all_buttons:
                try:
                    button_text = button.text.lower()
                    if "next" in button_text or ">" in button_text or "→" in button_text:
                        next_buttons = [button]
                        break
                except:
                    continue
        
        if not next_buttons:
            logger.info("No next page button found - reached the last page")
            return False
        
        # Check if the next button is enabled
        next_button = next_buttons[0]
        is_disabled = next_button.get_attribute("disabled") == "true" or "disabled" in next_button.get_attribute("class") or not next_button.is_enabled()
        
        if is_disabled:
            logger.info("Next page button is disabled - reached the last page")
            return False
        
        # Scroll to the button to make it visible
        self.driver.execute_script("arguments[0].scrollIntoView();", next_button)
        time.sleep(1)
        
        # Click the next button
        next_button.click()
        logger.info("Clicked next page button")
        
        # Wait for the page to load
        time.sleep(5)
        
        # For now, we'll assume the navigation was successful if we didn't get an error
        return True
        
    except Exception as e:
        logger.error(f"Error navigating to next page: {e}")
        return False

# Add the method to the class
AzurePartnerScraperSelenium.go_to_next_page = go_to_next_page

In [ ]:
def close(self) -> None:
    """Close the browser."""
    if self.driver:
        self.driver.quit()
        logger.info("Browser closed")

# Add the method to the class
AzurePartnerScraperSelenium.close = close

In [ ]:
# Test initializing the scraper
scraper = AzurePartnerScraperSelenium(headless=False)
print("Scraper initialized successfully")

In [ ]:
# Test loading the partners directory
scraper.load_partners_directory()
print("Partners directory loaded")

In [ ]:
# Test applying filters
# Uncomment and modify as needed
# scraper.apply_filter("location", "United States")
# scraper.apply_filter("capability", "AI")

In [ ]:
# Test extracting partners from the current page
partners = scraper.extract_partners_from_page()
print(f"Extracted {len(partners)} partners")

# Display the first few partners
if partners:
    for i, partner in enumerate(partners[:5]):
        print(f"\nPartner {i+1}:")
        print(f"  Name: {partner.name}")
        print(f"  URL: {partner.url}")
        print(f"  Location: {partner.location}")
        print(f"  Capabilities: {', '.join(partner.capabilities)}")
        print(f"  Description: {partner.description[:100]}..." if partner.description else "  Description: None")

In [ ]:
# Test going to the next page
has_next = scraper.go_to_next_page()
print(f"Navigation to next page {'successful' if has_next else 'failed'}")

# If navigation was successful, extract partners from the new page
if has_next:
    partners = scraper.extract_partners_from_page()
    print(f"Extracted {len(partners)} partners from the next page")

In [ ]:
# Function to save partners to JSON
def save_partners_to_json(partners, filename="extracted_partners.json"):
    """Save partners to a JSON file."""
    # Convert partners to dictionaries
    partners_data = [partner.to_dict() for partner in partners]
    
    # Save to file
    with open(filename, 'w') as f:
        json.dump(partners_data, f, indent=2)
    
    print(f"Saved {len(partners)} partners to {filename}")
    return filename

# Test saving partners
if partners:
    output_file = save_partners_to_json(partners)

In [ ]:
# Function to analyze partner data
def analyze_partner_data(partners):
    """Analyze partner data and display statistics."""
    if not partners:
        print("No partners to analyze")
        return
    
    # Count partners by location
    locations = {}
    for partner in partners:
        if partner.location:
            locations[partner.location] = locations.get(partner.location, 0) + 1
    
    # Count partners by capability
    capabilities = {}
    for partner in partners:
        for capability in partner.capabilities:
            capabilities[capability] = capabilities.get(capability, 0) + 1
    
    # Display statistics
    print(f"Total partners: {len(partners)}")
    
    print("\nTop locations:")
    for location, count in sorted(locations.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {location}: {count}")
    
    print("\nTop capabilities:")
    for capability, count in sorted(capabilities.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {capability}: {count}")
    
    # Create a DataFrame for more detailed analysis
    partner_dicts = [partner.to_dict() for partner in partners]
    df = pd.DataFrame(partner_dicts)
    
    # Display the DataFrame
    display(df.head())
    
    return df

# Test analyzing partner data
if partners:
    df = analyze_partner_data(partners)

In [ ]:
# Function to scrape multiple pages
def scrape_multiple_pages(scraper, max_pages=3, save_interval=1):
    """
    Scrape multiple pages of partners.
    
    Args:
        scraper: AzurePartnerScraperSelenium instance
        max_pages: Maximum number of pages to scrape
        save_interval: Interval at which to save partners
        
    Returns:
        List of PartnerData objects
    """
    all_partners = []
    page_count = 0
    
    try:
        while page_count < max_pages:
            # Extract partners from the current page
            page_partners = scraper.extract_partners_from_page()
            print(f"Page {page_count + 1}: Extracted {len(page_partners)} partners")
            
            # Add to the list of all partners
            all_partners.extend(page_partners)
            
            # Save partners at intervals
            if (page_count + 1) % save_interval == 0:
                save_partners_to_json(all_partners, f"partners_pages_1_to_{page_count + 1}.json")
            
            # Increment page count
            page_count += 1
            
            # Go to the next page
            if not scraper.go_to_next_page():
                print("No more pages available")
                break
            
            # Add a delay between page loads
            time.sleep(random.uniform(1.0, 3.0))
        
        print(f"Scraped {page_count} pages and found {len(all_partners)} partners")
        return all_partners
    
    except Exception as e:
        print(f"Error scraping multiple pages: {e}")
        # Save what we have so far
        if all_partners:
            save_partners_to_json(all_partners, f"partners_partial_{page_count + 1}_pages.json")
        return all_partners

# Uncomment to test multi-page scraping
# all_partners = scrape_multiple_pages(scraper, max_pages=3)

In [ ]:
# Function to apply filters and scrape
def apply_filters_and_scrape(scraper, filters, max_pages=3):
    """
    Apply filters and scrape partners.
    
    Args:
        scraper: AzurePartnerScraperSelenium instance
        filters: Dictionary of filters to apply
        max_pages: Maximum number of pages to scrape
        
    Returns:
        List of PartnerData objects
    """
    try:
        # Reload the partners directory to start fresh
        scraper.load_partners_directory()
        time.sleep(3)
        
        # Apply filters
        for filter_type, filter_value in filters.items():
            scraper.apply_filter(filter_type, filter_value)
            time.sleep(2)
        
        # Scrape multiple pages
        return scrape_multiple_pages(scraper, max_pages=max_pages)
    
    except Exception as e:
        print(f"Error applying filters and scraping: {e}")
        return []

# Example filters
filters = {
    # "location": "United States",
    # "capability": "AI",
    # "industry": "Healthcare"
}

# Uncomment to test applying filters and scraping
# filtered_partners = apply_filters_and_scrape(scraper, filters, max_pages=2)

In [ ]:
# Clean up resources
def cleanup():
    """Close the browser and clean up resources."""
    try:
        if 'scraper' in globals() and scraper:
            scraper.close()
            print("Browser closed")
    except Exception as e:
        print(f"Error closing browser: {e}")

# Uncomment to close the browser when done
# cleanup()

In [ ]:
# Function to run the complete pipeline
def run_complete_pipeline(headless=False, max_pages=3, filters=None):
    """
    Run the complete scraping pipeline.
    
    Args:
        headless: Whether to run the browser in headless mode
        max_pages: Maximum number of pages to scrape
        filters: Dictionary of filters to apply
        
    Returns:
        DataFrame of partner data
    """
    try:
        # Initialize the scraper
        scraper = AzurePartnerScraperSelenium(headless=headless)
        print("Scraper initialized")
        
        # Load the partners directory
        scraper.load_partners_directory()
        print("Partners directory loaded")
        
        # Apply filters if provided
        if filters:
            for filter_type, filter_value in filters.items():
                scraper.apply_filter(filter_type, filter_value)
                time.sleep(2)
            print("Filters applied")
        
        # Scrape multiple pages
        all_partners = scrape_multiple_pages(scraper, max_pages=max_pages)
        
        # Save all partners
        if all_partners:
            save_partners_to_json(all_partners, "all_partners.json")
        
        # Analyze partner data
        df = None
        if all_partners:
            df = analyze_partner_data(all_partners)
        
        # Clean up
        scraper.close()
        print("Pipeline completed successfully")
        
        return df
    
    except Exception as e:
        print(f"Error running pipeline: {e}")
        if 'scraper' in locals() and scraper:
            scraper.close()
        return None

# Uncomment to run the complete pipeline
# pipeline_filters = {
#     "location": "United States",
#     "capability": "AI"
# }
# partners_df = run_complete_pipeline(headless=False, max_pages=2, filters=pipeline_filters)

In [ ]:
# Create a simple interactive UI for testing
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def create_testing_ui():
    """Create an interactive UI for testing the scraper."""
    # Create widgets
    headless_checkbox = widgets.Checkbox(
        value=False,
        description='Headless Mode',
        disabled=False
    )
    
    max_pages_slider = widgets.IntSlider(
        value=2,
        min=1,
        max=10,
        step=1,
        description='Max Pages:',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d'
    )
    
    location_text = widgets.Text(
        value='',
        placeholder='Enter location (e.g., "United States")',
        description='Location:',
        disabled=False
    )
    
    capability_text = widgets.Text(
        value='',
        placeholder='Enter capability (e.g., "AI")',
        description='Capability:',
        disabled=False
    )
    
    industry_text = widgets.Text(
        value='',
        placeholder='Enter industry (e.g., "Healthcare")',
        description='Industry:',
        disabled=False
    )
    
    run_button = widgets.Button(
        description='Run Scraper',
        disabled=False,
        button_style='success',
        tooltip='Click to run the scraper',
        icon='play'
    )
    
    output = widgets.Output()
    
    # Define button click handler
    def on_run_button_clicked(b):
        with output:
            clear_output()
            print("Starting scraper...")
            
            # Collect filters
            filters = {}
            if location_text.value:
                filters['location'] = location_text.value
            if capability_text.value:
                filters['capability'] = capability_text.value
            if industry_text.value:
                filters['industry'] = industry_text.value
            
            # Run the pipeline
            run_complete_pipeline(
                headless=headless_checkbox.value,
                max_pages=max_pages_slider.value,
                filters=filters
            )
    
    # Connect the button click event
    run_button.on_click(on_run_button_clicked)
    
    # Display the UI
    display(widgets.VBox([
        widgets.HBox([headless_checkbox, max_pages_slider]),
        widgets.VBox([location_text, capability_text, industry_text]),
        run_button,
        output
    ]))

# Uncomment to create the interactive UI
# create_testing_ui()